# CellTypist Reference Annotation of Myeloid Cells

Transfers labels from pretrained CellTypist models onto the annotated myeloid object as an independent check of the marker-based `cell_state` labels:

- **HLCA** (`Human_Lung_Atlas`, healthy lung; Sikkema et al., Nat Med 2023): single-label, best match
- **Immune_All_High** (pan-tissue immune, low resolution; Domínguez Conde et al., Science 2022):
  multi-label, probability match (p > 0.5)

Both use majority voting over the stored neighbourhood graph. Predictions from each
model are kept in separate, prefixed columns and saved to `tables/`.

In [ ]:
# !pip install --quiet scanpy celltypist
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/myeloid

In [ ]:
import os

import celltypist
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from celltypist import models

## Config

In [ ]:
input_path = "sub_Myeloid_raw_scvi_combine_annotated.h5ad" 

figures_output_dir = "./figures"
tables_output_dir = "./tables"

counts_layer = "raw_counts"
reference_col = "cell_state"          # marker-based labels from step 2a_meyloid_fine_annotation
target_sum = 1e4                      # CellTypist expects log1p of counts normalised to 10,000

# one entry per model; columns are written as "<prefix>predicted_labels", "<prefix>majority_voting", ...
celltypist_runs = [
    {"prefix": "hlca_", "model": "Human_Lung_Atlas.pkl", "mode": "best match", "p_thres": 0.5,
     "highlight": ["Alveolar macrophages", "DC2", "Interstitial Mph perivascular"], "prob_umap": []},
    {"prefix": "immune_high_", "model": "Immune_All_High.pkl", "mode": "prob match", "p_thres": 0.5,
     "highlight": [], "prob_umap": ["Alveolar macrophages", "pDC"]},
    # {"prefix": "immune_low_", "model": "Immune_All_Low.pkl", "mode": "best match", "p_thres": 0.5,
    #  "highlight": [], "prob_umap": []},
]
majority_voting = True

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir
sc.settings.verbosity = 1
sc.set_figure_params(dpi=80, figsize=(4, 4))

## 1. Load and normalise

In [ ]:
adata = sc.read(input_path)

adata.X = adata.layers[counts_layer].copy() if counts_layer in adata.layers else adata.X.copy()
sample = adata.X[:1000].toarray() if sp.issparse(adata.X) else np.asarray(adata.X[:1000])
assert np.allclose(sample, np.round(sample)), "expected raw counts before normalisation"

adata.uns.pop("log1p", None)
sc.pp.normalize_total(adata, target_sum=target_sum)
sc.pp.log1p(adata)
assert "neighbors" in adata.uns, "majority voting over-clusters on the stored neighbourhood graph; run sc.pp.neighbors first"
print(adata.n_obs, "cells;", adata.obs[reference_col].nunique(), f"{reference_col} labels")

## 2. Load models

In [ ]:
loaded = {}
for run in celltypist_runs:
    models.download_models(model=run["model"], force_update=False)
    loaded[run["prefix"]] = models.Model.load(model=run["model"])
    print(run["model"], "|", loaded[run["prefix"]])

## 3. Annotate

`best match`: each cell gets the single highest-scoring label. `prob match`: each cell
gets every label with probability > `p_thres` (joined with `|`), or `Unassigned`.
Probabilities for all model cell types are stored as `<prefix><cell type>` columns.

In [ ]:
predictions = {}
for run in celltypist_runs:
    print(f"=== {run['model']} ({run['mode']})")
    pred = celltypist.annotate(adata, model=loaded[run["prefix"]], mode=run["mode"],
                               p_thres=run["p_thres"], majority_voting=majority_voting)
    adata = pred.to_adata(insert_prob=True, prefix=run["prefix"])
    predictions[run["prefix"]] = pred

## 4. Predictions on the UMAP

In [ ]:
for run in celltypist_runs:
    p = run["prefix"]
    sc.pl.umap(adata, color=[reference_col, f"{p}majority_voting", f"{p}predicted_labels"], ncols=3, wspace=0.6)
    for group in run["highlight"]:
        sc.pl.umap(adata, color=f"{p}predicted_labels", groups=[group], size=10, title=f"{run['model']}: {group}")
    prob_cols = [f"{p}{c}" for c in run["prob_umap"] if f"{p}{c}" in adata.obs]
    if prob_cols:
        sc.pl.umap(adata, color=[reference_col] + prob_cols, vmin=0, vmax=1)
    sc.pl.umap(adata, color=f"{p}conf_score", cmap="viridis", vmin=0, vmax=1)

## 5. Agreement with marker-based labels

Dotplots: for each `cell_state`, mean CellTypist probability (colour) and fraction of
cells (size) per predicted label. Tables: cell counts per `cell_state` × label.

In [ ]:
for run in celltypist_runs:
    p, pred = run["prefix"], predictions[run["prefix"]]
    tag = p.rstrip("_")
    for column, cmap in (("predicted_labels", "RdBu_r"), ("majority_voting", "Greens")):
        celltypist.dotplot(pred, use_as_reference=reference_col, use_as_prediction=column,
                           cmap=cmap, title=f"{run['model']}: {column}", save=f"_{tag}_{column}.pdf")
        table = pd.crosstab(adata.obs[reference_col], adata.obs[f"{p}{column}"])
        table.to_csv(f"{tables_output_dir}/celltypist_{tag}_{reference_col}_vs_{column}.csv")
    display(pd.crosstab(adata.obs[reference_col], adata.obs[f"{p}majority_voting"], normalize="index").round(2))

## 6. Save predictions

In [ ]:
label_cols = [c for run in celltypist_runs for c in
              (f"{run['prefix']}predicted_labels", f"{run['prefix']}over_clustering",
               f"{run['prefix']}majority_voting", f"{run['prefix']}conf_score") if c in adata.obs]
adata.obs[[reference_col] + label_cols].to_csv(f"{tables_output_dir}/celltypist_predictions.csv")

for run in celltypist_runs:
    p = run["prefix"]
    prob_cols = [f"{p}{c}" for c in loaded[p].cell_types if f"{p}{c}" in adata.obs]
    adata.obs[prob_cols].to_csv(f"{tables_output_dir}/celltypist_{p.rstrip('_')}_probabilities.csv.gz")
print("saved:", label_cols)